In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 1: Databricks Environment Inspection

print("=== CURRENT CATALOG ===")
spark.sql("SELECT current_catalog()").show()

print("=== CURRENT SCHEMA ===")
spark.sql("SELECT current_schema()").show()

print("=== AVAILABLE TABLES ===")
spark.sql("SHOW TABLES").show(100, truncate=False)

print("=== SPARK VERSION ===")
print(spark.version)

=== CURRENT CATALOG ===
+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+

=== CURRENT SCHEMA ===
+----------------+
|current_schema()|
+----------------+
|         default|
+----------------+

=== AVAILABLE TABLES ===
+--------+-------------------------------------+-----------+
|database|tableName                            |isTemporary|
+--------+-------------------------------------+-----------+
|default |applebees_analysis_master            |false      |
|default |applebees_discount_events            |false      |
|default |applebees_outliers                   |false      |
|default |applebees_payment_events             |false      |
|default |applebees_void_events                |false      |
|default |employee_risk_scores                 |false      |
|default |p2_bronze_checks                     |false      |
|default |p2_bronze_employees                  |false      |
|default |p2_bronze_items                      |false      |

In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 2: Existing Tables Inspection
# Permission-safe inspection

tables = [
    "applebees_analysis_master",
    "applebees_discount_events",
    "applebees_outliers",
    "applebees_payment_events",
    "applebees_void_events",
    "employee_risk_scores"
]

print("=== EXISTING TABLES INSPECTION ===")

inspection_results = []

for table in tables:

    full_name = f"workspace.default.{table}"

    print("\n" + "=" * 65)
    print(f"TABLE: {full_name}")
    print("=" * 65)

    try:
        df = spark.table(full_name)

        print("\nCOLUMNS:")
        print(df.columns)

        print("\nSTORAGE INFORMATION:")

        details = spark.sql(
            f"DESCRIBE DETAIL {full_name}"
        )

        details.select(
            "format",
            "location",
            "numFiles",
            "sizeInBytes"
        ).show(truncate=False)

        print("STATUS: PASS")

        inspection_results.append({
            "table": table,
            "status": "PASS"
        })

    except Exception as e:

        print("STATUS: SKIPPED")
        print("Reason:", str(e).splitlines()[0])

        inspection_results.append({
            "table": table,
            "status": "SKIPPED"
        })

print("\n=== INSPECTION SUMMARY ===")

for result in inspection_results:
    print(
        f"{result['status']} | {result['table']}"
    )

print("\nINSPECTION COMPLETED")

=== EXISTING TABLES INSPECTION ===

TABLE: workspace.default.applebees_analysis_master

COLUMNS:
['LocationId', 'BusinessDate', 'CheckId', 'RevenueCenterId', 'CloseTime', 'Discounts', 'ItemsSold', 'OpenTime', 'Payments', 'ServiceCharge', 'TableName', 'Tax', 'TrafficCount', 'Voids', 'flag_zero_duration', 'flag_no_items', 'flag_no_payments', 'flag_empty_check', 'LocationName', 'OpenDate', 'ClosedDate', 'TimeZoneName', 'RevenueCenterName', 'total_item_quantity', 'total_sold_price', 'item_event_count', 'item_employee_count', 'discount_count', 'discount_amount', 'max_discount', 'discount_employee_count', 'payment_count', 'payment_amount', 'gratuity_raw', 'payment_employee_count', 'void_count', 'void_amount', 'max_void', 'void_employee_count']

STORAGE INFORMATION:
+------+--------+--------+-----------+
|format|location|numFiles|sizeInBytes|
+------+--------+--------+-----------+
|delta |        |6       |351863770  |
+------+--------+--------+-----------+

STATUS: PASS

TABLE: workspace.def

In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 2: Table Access & Permissions Audit

tables = [
    "applebees_analysis_master",
    "applebees_discount_events",
    "applebees_outliers",
    "applebees_payment_events",
    "applebees_void_events",
    "employee_risk_scores"
]

results = []

for table in tables:
    full_name = f"workspace.default.{table}"

    try:
        df = spark.table(full_name)

        # Verify actual read access
        df.limit(1).collect()

        results.append({
            "Table": table,
            "Access": "AVAILABLE",
            "Columns": len(df.columns),
            "Details": "Read successful"
        })

    except Exception as e:
        results.append({
            "Table": table,
            "Access": "RESTRICTED / ERROR",
            "Columns": None,
            "Details": str(e).splitlines()[0][:180]
        })

display(spark.createDataFrame(results))

Access,Columns,Details,Table
AVAILABLE,39,Read successful,applebees_analysis_master
AVAILABLE,9,Read successful,applebees_discount_events
RESTRICTED / ERROR,null,[INSUFFICIENT_PERMISSIONS] Insufficient privileges:,applebees_outliers
AVAILABLE,17,Read successful,applebees_payment_events
AVAILABLE,13,Read successful,applebees_void_events
RESTRICTED / ERROR,null,[INSUFFICIENT_PERMISSIONS] Insufficient privileges:,employee_risk_scores


In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 3: Catalog and Schema Inspection

print("=== CURRENT USER ===")
spark.sql("SELECT current_user()").show(truncate=False)

print("=== AVAILABLE SCHEMAS ===")
spark.sql("SHOW SCHEMAS IN workspace").show(
    truncate=False
)

print("=== CURRENT SCHEMA DETAILS ===")
spark.sql(
    "DESCRIBE SCHEMA EXTENDED workspace.default"
).show(truncate=False)

print("=== EXISTING TABLE FORMATS ===")

for table in [
    "applebees_analysis_master",
    "applebees_discount_events",
    "applebees_payment_events",
    "applebees_void_events"
]:
    try:
        print(f"\nTable: {table}")

        spark.sql(
            f"DESCRIBE DETAIL workspace.default.{table}"
        ).select(
            "format",
            "numFiles",
            "sizeInBytes"
        ).show(truncate=False)

    except Exception as e:
        print(f"Metadata unavailable: {str(e)[:150]}")

=== CURRENT USER ===
+----------------+
|current_user()  |
+----------------+
|a01739136@tec.mx|
+----------------+

=== AVAILABLE SCHEMAS ===
+------------------+
|databaseName      |
+------------------+
|default           |
|information_schema|
+------------------+

=== CURRENT SCHEMA DETAILS ===
+-------------------------+---------------------------------------------------------+
|database_description_item|database_description_value                               |
+-------------------------+---------------------------------------------------------+
|Catalog Name             |workspace                                                |
|Namespace Name           |default                                                  |
|Comment                  |Default schema (auto-created)                            |
|Location                 |                                                         |
|Owner                    |_workspace_admins_workspace_7474654356139945             |
|Properties

In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 4: Permissions and Storage Inspection

print("=== CATALOG PRIVILEGES ===")

try:
    spark.sql(
        "SHOW GRANTS ON CATALOG workspace"
    ).show(100, truncate=False)

except Exception as e:
    print("Grant details unavailable:")
    print(str(e).splitlines()[0][:250])


print("\n=== SCHEMA PRIVILEGES ===")

try:
    spark.sql(
        "SHOW GRANTS ON SCHEMA workspace.default"
    ).show(100, truncate=False)

except Exception as e:
    print("Grant details unavailable:")
    print(str(e).splitlines()[0][:250])


print("\n=== AVAILABLE VOLUMES ===")

try:
    spark.sql(
        "SHOW VOLUMES IN workspace.default"
    ).show(truncate=False)

except Exception as e:
    print("Volume details unavailable:")
    print(str(e).splitlines()[0][:250])


print("\n=== CURRENT WORKSPACE PATH ===")

try:
    print(
        dbutils.notebook.entry_point
        .getDbutils()
        .notebook()
        .getContext()
        .notebookPath()
        .get()
    )

except Exception as e:
    print("Notebook path unavailable:")
    print(str(e).splitlines()[0][:250])

=== CATALOG PRIVILEGES ===
+-------------------------------------------+-----------+----------+---------+
|Principal                                  |ActionType |ObjectType|ObjectKey|
+-------------------------------------------+-----------+----------+---------+
|a01739136@tec.mx                           |USE CATALOG|CATALOG   |workspace|
|a01739136@tec.mx                           |USE SCHEMA |CATALOG   |workspace|
|account users                              |USE CATALOG|CATALOG   |workspace|
|_workspace_users_workspace_7474654356139945|USE CATALOG|CATALOG   |workspace|
+-------------------------------------------+-----------+----------+---------+


=== SCHEMA PRIVILEGES ===
+-------------------------------------------+------------------------+----------+-----------------+
|Principal                                  |ActionType              |ObjectType|ObjectKey        |
+-------------------------------------------+------------------------+----------+-----------------+
|a01739136@te

In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 5: Raw Data Volume Inspection

raw_path = "/Volumes/workspace/default/applebees_datachefs_raw/"

print("=== RAW DATA VOLUME ===")
print(raw_path)

try:
    files = dbutils.fs.ls(raw_path)

    print("\n=== AVAILABLE FILES / FOLDERS ===")

    for file in files[:50]:
        print(
            f"Name: {file.name} | "
            f"Size: {file.size} bytes | "
            f"Path: {file.path}"
        )

    print(f"\nTotal entries at root: {len(files)}")

except Exception as e:
    print("Unable to access volume:")
    print(str(e).splitlines()[0][:300])

=== RAW DATA VOLUME ===
/Volumes/workspace/default/applebees_datachefs_raw/

=== AVAILABLE FILES / FOLDERS ===
Name: OneDrive_1_5-10-2026/ | Size: 0 bytes | Path: dbfs:/Volumes/workspace/default/applebees_datachefs_raw/OneDrive_1_5-10-2026/

Total entries at root: 1


In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 6: Raw Source Files Discovery

from collections import Counter

source_path = (
    "/Volumes/workspace/default/"
    "applebees_datachefs_raw/"
    "OneDrive_1_5-10-2026/"
)

all_files = []
folders = []

def explore_folder(path, depth=0, max_depth=5):

    if depth > max_depth:
        return

    try:
        entries = dbutils.fs.ls(path)

        for entry in entries:

            if entry.isDir():
                folders.append(entry.path)

                print(
                    "  " * depth +
                    f"📁 {entry.name}"
                )

                explore_folder(
                    entry.path,
                    depth + 1,
                    max_depth
                )

            else:
                all_files.append({
                    "name": entry.name,
                    "path": entry.path,
                    "size_bytes": entry.size,
                    "extension": (
                        entry.name.rsplit(".", 1)[-1].lower()
                        if "." in entry.name
                        else "unknown"
                    )
                })

    except Exception as e:
        print(f"Cannot access {path}: {str(e)[:150]}")


print("=== SOURCE DIRECTORY STRUCTURE ===")

explore_folder(source_path)

print("\n=== FILE INVENTORY ===")
print(f"Total files: {len(all_files)}")
print(f"Total folders: {len(folders)}")

print("\n=== FILE FORMATS ===")

format_counts = Counter(
    f["extension"] for f in all_files
)

for extension, count in format_counts.items():
    print(f"{extension}: {count}")

print("\n=== SAMPLE FILES ===")

for file in all_files[:15]:
    print(file["path"])

=== SOURCE DIRECTORY STRUCTURE ===
📁 dimensions/
  📁 employees/
  📁 items/
  📁 locations/
  📁 revenue_centers/
📁 facts/
  📁 checks/
    📁 brand=rmh/
      📁 dt=2026-04-28/
      📁 dt=2026-04-29/
      📁 dt=2026-04-30/
      📁 dt=2026-05-01/
      📁 dt=2026-05-02/
      📁 dt=2026-05-03/
      📁 dt=2026-05-04/
      📁 dt=2026-05-05/
      📁 dt=2026-05-06/
      📁 dt=2026-05-07/
      📁 dt=2026-05-08/
      📁 dt=2026-05-09/
      📁 dt=2026-05-10/
      📁 dt=2026-05-11/
      📁 dt=2026-05-12/
      📁 dt=2026-05-13/
      📁 dt=2026-05-14/
      📁 dt=2026-05-15/
      📁 dt=2026-05-16/
      📁 dt=2026-05-17/
      📁 dt=2026-05-18/
      📁 dt=2026-05-19/
      📁 dt=2026-05-20/
      📁 dt=2026-05-21/
      📁 dt=2026-05-22/
      📁 dt=2026-05-23/
      📁 dt=2026-05-24/
      📁 dt=2026-05-25/
      📁 dt=2026-05-26/
      📁 dt=2026-05-27/
      📁 dt=2026-05-28/
      📁 dt=2026-05-29/
      📁 dt=2026-05-30/
      📁 dt=2026-05-31/
      📁 dt=2026-06-01/
      📁 dt=2026-06-02/
      📁 dt=2026-06-03/


In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 7: Source Schema Validation

base_path = (
    "/Volumes/workspace/default/"
    "applebees_datachefs_raw/"
    "OneDrive_1_5-10-2026/"
)

checks_path = (
    base_path +
    "facts/checks/brand=rmh/"
    "dt=2026-04-28/checks.json"
)

employees_path = (
    base_path +
    "dimensions/employees/employees.json"
)

sources = {
    "checks_sample": checks_path,
    "employees": employees_path
}

for source_name, path in sources.items():

    print(f"\n{'='*60}")
    print(f"SOURCE: {source_name}")
    print(f"{'='*60}")

    try:
        df = (
            spark.read
            .option("multiLine", "true")
            .json(path)
        )

        print("Read status: SUCCESS")
        print("Columns:", df.columns)
        print("Schema:")
        df.printSchema()

    except Exception as e:
        print("Read status: FAILED")
        print(str(e).splitlines()[0][:300])


SOURCE: checks_sample
Read status: SUCCESS
Columns: ['BusinessDate', 'CloseTime', 'Discounts', 'Id', 'ItemsSold', 'LocationId', 'LoyaltyId', 'OpenTime', 'Payments', 'RevenueCenterId', 'ServiceCharge', 'TableName', 'Tax', 'TrafficCount', 'Voids']
Schema:
root
 |-- BusinessDate: string (nullable = true)
 |-- CloseTime: string (nullable = true)
 |-- Discounts: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Amount: double (nullable = true)
 |    |    |-- DiscountId: long (nullable = true)
 |    |    |-- DiscountName: string (nullable = true)
 |    |    |-- EmployeeId: long (nullable = true)
 |    |    |-- ManagerId: long (nullable = true)
 |-- Id: long (nullable = true)
 |-- ItemsSold: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discounts: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- Amount: double (nullable = true)
 |    |    |    |    |-- Disco

In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 8: JSON Reading Strategy Validation

from pyspark.sql import functions as F

base_path = (
    "/Volumes/workspace/default/"
    "applebees_datachefs_raw/"
    "OneDrive_1_5-10-2026/"
)

sources = {
    "checks_2026_04_28": (
        base_path +
        "facts/checks/brand=rmh/"
        "dt=2026-04-28/checks.json"
    ),
    "employees": (
        base_path +
        "dimensions/employees/employees.json"
    )
}

results = []

for source_name, path in sources.items():

    for multiline in [True, False]:

        try:
            df = (
                spark.read
                .option("multiLine", str(multiline).lower())
                .json(path)
            )

            results.append({
                "source": source_name,
                "multiLine": str(multiline),
                "rows": df.count(),
                "columns": len(df.columns),
                "corrupt_column": (
                    "_corrupt_record" in df.columns
                ),
                "status": "SUCCESS"
            })

        except Exception as e:

            results.append({
                "source": source_name,
                "multiLine": str(multiline),
                "rows": None,
                "columns": None,
                "corrupt_column": None,
                "status": str(e).splitlines()[0][:150]
            })

display(spark.createDataFrame(results))

columns,corrupt_column,multiLine,rows,source,status
15,false,True,15467,checks_2026_04_28,SUCCESS
15,false,False,15467,checks_2026_04_28,SUCCESS
15,false,True,156975,employees,SUCCESS
1,true,False,3492975,employees,SUCCESS


In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 9: Complete Source Validation

from pyspark.sql import functions as F

base = (
    "/Volumes/workspace/default/"
    "applebees_datachefs_raw/"
    "OneDrive_1_5-10-2026/"
)

sources = {
    "employees": "dimensions/employees/employees.json",
    "items": "dimensions/items/items.json",
    "locations": "dimensions/locations/locations.json",
    "revenue_centers": "dimensions/revenue_centers/revenueCenters.json"
}

print("=== DIMENSION VALIDATION ===")

for name, relative_path in sources.items():
    try:
        df = (
            spark.read
            .option("multiLine", "true")
            .json(base + relative_path)
        )

        print(
            f"{name}: "
            f"{df.count():,} rows | "
            f"{len(df.columns)} columns"
        )

    except Exception as e:
        print(f"{name}: ERROR - {str(e)[:180]}")


print("\n=== CHECKS VALIDATION ===")

try:
    checks_path = base + "facts/checks/brand=rmh/"

    checks_df = (
        spark.read
        .option("multiLine", "true")
        .option("recursiveFileLookup", "true")
        .json(checks_path)
    )

    print("Total checks:", f"{checks_df.count():,}")

    print(
        "Source files:",
        checks_df.select(
            F.input_file_name().alias("source_file")
        ).distinct().count()
    )

    print("Columns:", len(checks_df.columns))

except Exception as e:
    print("Checks validation error:")
    print(str(e)[:300])

=== DIMENSION VALIDATION ===
employees: 156,975 rows | 15 columns
items: 16,697 rows | 5 columns
locations: 185 rows | 5 columns
revenue_centers: 23 rows | 2 columns

=== CHECKS VALIDATION ===
Total checks: 2,142,846
Checks validation error:
[UC_COMMAND_NOT_SUPPORTED.WITH_RECOMMENDATION] The command(s): input_file_name are not supported in Unity Catalog. Please use _metadata.file_path instead. SQLSTATE: 0AKUC;
Aggregate [count(1) AS count(1)#115649L]
+- Deduplicate [source_file#115645]
   +- Project [input_file_name() AS source_file#115


In [0]:
# PHASE 2 — ARCHITECTURE & JOB
# Step 10: Source File and Date Validation

from pyspark.sql import functions as F

base = (
    "/Volumes/workspace/default/"
    "applebees_datachefs_raw/"
    "OneDrive_1_5-10-2026/"
)

checks_path = base + "facts/checks/brand=rmh/"

checks_df = (
    spark.read
    .option("multiLine", "true")
    .option("recursiveFileLookup", "true")
    .json(checks_path)
    .select(
        "Id",
        "LocationId",
        "BusinessDate",
        F.col("_metadata.file_path").alias("source_file")
    )
)

summary = checks_df.agg(
    F.count("*").alias("total_checks"),
    F.countDistinct("source_file").alias("source_files"),
    F.countDistinct("BusinessDate").alias("business_dates"),
    F.min("BusinessDate").alias("first_business_date"),
    F.max("BusinessDate").alias("last_business_date")
)

print("=== SOURCE VALIDATION SUMMARY ===")
display(summary)

print("=== SAMPLE SOURCE FILES ===")

display(
    checks_df
    .select("source_file")
    .distinct()
    .limit(5)
)

=== SOURCE VALIDATION SUMMARY ===


total_checks,source_files,business_dates,first_business_date,last_business_date
2142846,115,115,2026-04-28,2026-08-20


=== SAMPLE SOURCE FILES ===


source_file
dbfs:/Volumes/workspace/default/applebees_datachefs_raw/OneDrive_1_5-10-2026/facts/checks/brand=rmh/dt=2026-05-22/checks.json
dbfs:/Volumes/workspace/default/applebees_datachefs_raw/OneDrive_1_5-10-2026/facts/checks/brand=rmh/dt=2026-05-29/checks.json
dbfs:/Volumes/workspace/default/applebees_datachefs_raw/OneDrive_1_5-10-2026/facts/checks/brand=rmh/dt=2026-07-31/checks.json
dbfs:/Volumes/workspace/default/applebees_datachefs_raw/OneDrive_1_5-10-2026/facts/checks/brand=rmh/dt=2026-05-10/checks.json
dbfs:/Volumes/workspace/default/applebees_datachefs_raw/OneDrive_1_5-10-2026/facts/checks/brand=rmh/dt=2026-06-21/checks.json


# Phase 2 — Pipeline Architecture & Orchestration

## Business Objective

Build a reproducible Databricks analytics pipeline to identify unusual transaction patterns related to discounts, voids, employees, locations, and transaction behavior. Risk indicators represent potential anomalies and do not constitute confirmed fraud.

## Source Layer

- **Storage:** Unity Catalog Volume
- **Path:** `/Volumes/workspace/default/applebees_datachefs_raw/OneDrive_1_5-10-2026/`
- **Format:** JSON
- **Source files:** 119 (115 daily transaction files and 4 dimension files)
- **Transaction records:** 2,142,846
- **Business dates:** April 28 to August 20, 2026
- **Reading strategy:** `multiLine=true`

## Bronze Layer — Implemented and Validated

Five Delta tables preserve the source records and dimensions, including nested transaction data and ingestion metadata.

All five tables were successfully created and verified under the `p2_team_` prefix.

## Silver Layer — Implemented and Validated

Five Delta tables standardize transaction identifiers, structure transaction events, calculate financial measures, and retain anomaly indicators.

The Silver checks table contains 2,142,846 records, matching the Bronze checks count.

The current implementation includes five Silver tables rather than the ten originally proposed in the architecture.

## Data Quality Gate — Pending Integration

The planned Quality Gate will evaluate critical data quality rules before Gold execution.

The Data Quality notebook is being developed separately and has not yet been integrated into the automated workflow.

## Gold Layer — Implemented and Validated

Four Delta tables support transaction-level risk flags, daily risk metrics, location comparisons, and employee risk profiles.

All four tables were created and verified. Transaction identifiers and selected financial measures were successfully reconciled between Silver and Gold.

## Workflow Orchestration

**Current automated workflow:**

`preflight_sources → bronze_ingestion → silver_transformations → gold_risk_analytics`

The four-task workflow executed successfully on October 29, 2026, with a duration of 3 minutes and 9 seconds.

**Target workflow:**

`preflight_sources → bronze_ingestion → silver_transformations → data_quality_gate → gold_risk_analytics → business_insights`

The final workflow will require successful completion of the Data Quality Gate before Gold execution.

## Engineering Validation

- Source accessibility: passed for all five source datasets.
- Automated workflow execution: 4/4 tasks succeeded.
- Row-count stability: 14/14 tables passed.
- Transaction key integrity: passed across Bronze, Silver, and Gold.
- Cross-layer identifier consistency: passed between Silver and Gold.
- Financial consistency: 4/4 shared financial measures passed.
- Controlled upstream failure blocking: passed.

## Current Status

Bronze, Silver, and Gold transformations are implemented and have been successfully executed through Databricks Workflows.

The remaining integration activities are the Data Quality Gate, Business Insights, and final end-to-end workflow validation.

Full idempotency and comprehensive data quality compliance have not yet been established.


flowchart TD
    A["Unity Catalog Volume<br/>119 JSON files"]
    B["Preflight<br/>Source Access Validation"]
    C["Bronze Layer<br/>5 Delta Tables"]
    D["Silver Layer<br/>5 Delta Tables"]
    E{"Data Quality Gate<br/>Pending Integration"}
    F["Gold Layer<br/>4 Delta Tables"]
    G["Business Insights<br/>Pending Integration"]

    A --> B
    B --> C
    C --> D
    D --> E
    E -->|Pass| F
    E -->|Fail| H["Stop Pipeline<br/>Quality Failure"]
    F --> G

    classDef complete fill:#dbeafe,stroke:#2563eb,color:#1e3a8a
    classDef pending fill:#fef3c7,stroke:#d97706,color:#78350f
    classDef failure fill:#fee2e2,stroke:#dc2626,color:#7f1d1d

    class A,B,C,D,F complete
    class E,G pending
    class H failure


## Failure Handling & Recovery Strategy

The Applebee's Fraud Risk Analytics Pipeline uses a fail-fast orchestration strategy through Databricks Workflows.

### Failure Prevention and Dependency Management

1. **Source preflight:** The `preflight_sources` task verifies accessibility to the five source datasets before ingestion.
2. **Sequential execution:** Downstream tasks require successful completion of their upstream dependencies.
3. **Failure blocking:** When an upstream task fails, dependent tasks are prevented from executing.
4. **Retries:** Transient failures may be retried according to the configured task settings.
5. **Data protection:** The Phase 2 implementation uses independent `p2_team_*` Delta tables to avoid modifying the original Phase 1 tables.
6. **Data Quality Gate:** A critical quality failure is intended to block Gold execution once the gate is integrated.

### Controlled Failure Test

An isolated Databricks Job was used to test failure propagation.

| Component | Result |
|---|---|
| Test Job ID | `97668685275602` |
| Run ID | `464075315429620` |
| `simulated_failure` | FAILED — intentionally |
| `blocked_downstream` | UPSTREAM FAILED |
| Downstream execution | Blocked |
| Failed task attempts | 2 |

**Result: PASSED**

The controlled test confirmed that Databricks Workflows prevented the dependent task from executing after an upstream failure.

The observed two attempts provide evidence of repeated execution attempts, although the final retry configuration and recovery behavior still require verification.

### Recovery Procedure

1. Identify the failed task in Databricks Jobs & Pipelines.
2. Review task output, execution logs, and error messages.
3. Determine whether the failure was caused by permissions, unavailable source files, transformation logic, or data quality.
4. Correct the underlying issue.
5. Verify that the affected task can be safely rerun without introducing duplicate records or unintended changes.
6. Use the appropriate repair or rerun mechanism.
7. Revalidate downstream table counts, transaction keys, and relevant financial measures.

### Current Implementation Status

**Implemented and tested:**
- Source preflight validation.
- Sequential task dependencies.
- Successful four-task automated execution.
- Controlled upstream failure blocking.
- Stable row counts across 14 Delta tables.
- Transaction key integrity and selected financial consistency checks.

**Pending:**
- Integration and failure testing of the Data Quality Gate.
- Recovery testing on the complete analytical workflow.
- Final verification of retry settings and scheduling.
- End-to-end execution including Business Insights.

**Conclusion:** The current workflow demonstrates successful orchestration and controlled failure blocking. Full production-style recovery and data quality enforcement remain pending final team integration.

## Workflow Validation & Test Evidence

### 1. Automated Workflow Execution

The Applebee's Fraud Risk Analytics Pipeline was successfully executed through Databricks Workflows on October 29, 2026.

**Job ID:** `783191776406860`  
**Successful Run ID:** `687046371295810`  
**Execution duration:** 3 minutes and 9 seconds  
**Execution environment:** Databricks Serverless

| Task | Execution result |
|---|---|
| `preflight_sources` | SUCCEEDED |
| `bronze_ingestion` | SUCCEEDED |
| `silver_transformations` | SUCCEEDED |
| `gold_risk_analytics` | SUCCEEDED |

**Result: PASSED — 4/4 tasks completed successfully.**

The workflow demonstrated sequential execution of source validation, Bronze ingestion, Silver transformations, and Gold analytics.

### 2. Delta Table Validation

A total of 14 Delta tables were created and verified under the `workspace.default.p2_team_*` naming convention.

| Layer | Tables verified | Result |
|---|---:|---|
| Bronze | 5/5 | PASS |
| Silver | 5/5 | PASS |
| Gold | 4/4 | PASS |
| **Total** | **14/14** | **PASS** |

All table row counts matched the reference results from the original transformation implementation.

### 3. Row-Count Stability

Following a subsequent automated pipeline execution, all 14 tables were compared against their previously recorded baseline counts.

**Results:**
- Bronze: 5/5 unchanged.
- Silver: 5/5 unchanged.
- Gold: 4/4 unchanged.
- Total: 14/14 unchanged.

**Result: PASSED — No row-count differences detected.**

This supports repeatability under the tested conditions but does not independently establish complete idempotency.

### 4. Transaction Key Integrity

Transaction

In [0]:
print("=== CHECK TABLE CREATION PRIVILEGES ===")

spark.sql("""
SHOW GRANTS ON SCHEMA workspace.default
""").show(truncate=False)

=== CHECK TABLE CREATION PRIVILEGES ===
+-------------------------------------------+------------------------+----------+-----------------+
|Principal                                  |ActionType              |ObjectType|ObjectKey        |
+-------------------------------------------+------------------------+----------+-----------------+
|a01739136@tec.mx                           |USE SCHEMA              |CATALOG   |workspace        |
|account users                              |USE SCHEMA              |SCHEMA    |workspace.default|
|_workspace_users_workspace_7474654356139945|CREATE TABLE            |SCHEMA    |workspace.default|
|_workspace_users_workspace_7474654356139945|CREATE VOLUME           |SCHEMA    |workspace.default|
|_workspace_users_workspace_7474654356139945|CREATE FUNCTION         |SCHEMA    |workspace.default|
|_workspace_users_workspace_7474654356139945|CREATE MODEL            |SCHEMA    |workspace.default|
|_workspace_users_workspace_7474654356139945|USE SCHEMA     

In [0]:
test_table = "workspace.default.p2_permission_test"

try:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {test_table}
        USING DELTA
        AS SELECT 1 AS test_value
    """)

    result = spark.table(test_table).collect()

    print("CREATE TABLE: PASSED")
    print("SELECT: PASSED")
    print(f"Test result: {result[0]['test_value']}")

except Exception as e:
    print("PERMISSION TEST: FAILED")
    print(str(e)[:1200])

CREATE TABLE: PASSED
SELECT: PASSED
Test result: 1


In [0]:
print("=== RAW SOURCE ACCESS TEST ===")

base = (
    "/Volumes/workspace/default/applebees_datachefs_raw/"
    "OneDrive_1_5-10-2026/"
)

sources = {
    "checks": base + "facts/checks/brand=rmh/",
    "employees": base + "dimensions/employees/employees.json",
    "items": base + "dimensions/items/items.json",
    "locations": base + "dimensions/locations/locations.json",
    "revenue_centers": base + "dimensions/revenue_centers/revenueCenters.json"
}

for name, path in sources.items():
    try:
        df = (
            spark.read
            .option("multiLine", "true")
            .json(path)
        )

        df.limit(1).collect()

        print(f"PASS | {name} | JSON readable")

    except Exception as e:
        print(f"BLOCKED | {name}")
        print(str(e).splitlines()[0])

print("=== TEST COMPLETED ===")

=== RAW SOURCE ACCESS TEST ===
PASS | checks | JSON readable
PASS | employees | JSON readable
PASS | items | JSON readable
PASS | locations | JSON readable
PASS | revenue_centers | JSON readable
=== TEST COMPLETED ===


### Pipeline Execution and Row-Count Stability Validation

The Applebee's Fraud Risk Analytics Pipeline was successfully executed through Databricks Workflows using four sequential tasks: `preflight_sources`, `bronze_ingestion`, `silver_transformations`, and `gold_risk_analytics`.

The automated run completed successfully on October 29, 2026, in 3 minutes and 9 seconds (Run ID: `687046371295810`). All four tasks returned a successful status.

To assess repeatability, row counts were recorded for 14 Delta tables before the subsequent pipeline execution and compared with the counts afterward. All 14 tables retained their original row counts: five Bronze tables, five Silver tables, and four Gold tables. No row-count differences were detected.

**Result:** Row-count stability test passed (14/14). This provides evidence of repeatable execution without row-count growth under the tested input conditions. Full idempotency validation remains pending because row counts alone do not establish record-level equality or the absence of duplicate keys.

**Pending integration:** The Data Quality Gate must be inserted between Silver and Gold before the final production-style workflow validation.

### Data Integrity and Financial Consistency Validation

To evaluate the reliability of the Bronze, Silver, and Gold transformations, additional integrity and consistency tests were performed on the Applebee's transaction dataset.

**1. Transaction Key Integrity**

The Bronze layer was validated using the composite key `LocationId + BusinessDate + Id`. All 2,142,846 records passed the validation, with zero null composite keys and zero duplicate key groups.

In Silver and Gold, the standardized identifier `check_key` was used to validate transaction uniqueness and consistency across layers.

- Silver: 0 null keys, 0 duplicate key groups, and 0 flagged duplicates.
- Gold: 0 null keys, 0 duplicate key groups, and 0 flagged duplicates.
- Cross-layer comparison: 0 Silver keys missing in Gold and 0 additional Gold keys.

**Result:** Transaction key integrity validation passed.

**2. Financial Consistency**

Four financial variables were compared between Silver and Gold using `check_key` as the transaction identifier:

| Variable | Records with relevant differences |
|---|---:|
| `total_gross_price` | 0 |
| `total_sold_price` | 0 |
| `payment_amount` | 0 |
| `void_amount` | 0 |

The comparison applied a numerical tolerance of 0.01 and checked for null-value inconsistencies.

**Result:** Financial consistency validation passed (4/4 variables).

**3. Pipeline Repeatability**

The automated Databricks Workflow successfully executed four sequential tasks: `preflight_sources`, `bronze_ingestion`, `silver_transformations`, and `gold_risk_analytics`.

Following a subsequent execution, all 14 Delta tables maintained their baseline row counts:

- Bronze: 5/5 tables unchanged.
- Silver: 5/5 tables unchanged.
- Gold: 4/4 tables unchanged.

**Result:** Row-count stability validation passed (14/14 tables).

**Overall Assessment**

The results provide evidence of transaction key integrity, financial consistency between Silver and Gold, and stable record counts across repeated pipeline executions.

However, full idempotency and comprehensive data quality validation have not yet been established. The Data Quality Gate remains pending integration between Silver and Gold, followed by final end-to-end workflow testing.